# 01 — Bronze: ingestão dos arquivos e da PTAX

**Objetivo:** preservar os valores originais dos CSVs, acrescentar `ingestion_datetime` e gravar em Delta com **append**.

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import re

hoje = datetime.now(ZoneInfo("America/Sao_Paulo")).date()
def parametro(nome, padrao):
    try:
        return dbutils.widgets.get(nome)
    except Exception:
        dbutils.widgets.text(nome, padrao)
        return dbutils.widgets.get(nome)

CATALOGO = parametro("catalogo", "workspace")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOGO):
    raise ValueError("Use um nome simples de catálogo, com letras, números e underscore.")
DATA_INICIO = datetime.strptime(parametro("data_inicio", (hoje-timedelta(days=6)).strftime("%m-%d-%Y")), "%m-%d-%Y").date()
DATA_FIM = datetime.strptime(parametro("data_fim", hoje.strftime("%m-%d-%Y")), "%m-%d-%Y").date()
if DATA_INICIO > DATA_FIM or DATA_FIM > hoje:
    raise ValueError("O período deve estar em ordem e não pode terminar no futuro.")

def tabela(camada, nome):
    return f"{CATALOGO}.{camada}.{nome}"

CAMINHO_INPUTS = parametro("caminho_inputs", f"/Volumes/{CATALOGO}/default/inputs").rstrip("/")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.bronze")
arquivos_disponiveis = {item.name.rstrip("/") for item in dbutils.fs.ls(CAMINHO_INPUTS)}

mapeamento = {
    "tb_movies_info": ["movies_info_TMDB_IMDB.csv", "movies_info_IMDB_TMDB.csv"],
    "tb_movies_financials": ["movies_financials_IMDB_TMDB.csv"],
    "tb_movies_metrics": ["movies_metrics_IMDB_TMDB.csv", "movies_metics_IMDB_TMDB.csv"],
    "tb_credits_and_tags": ["credits_and_tags_IMDB_TMDB.csv"],
    "tb_movies_reviews": ["movies_reviews.csv"]
}
for nome_tabela, candidatos in mapeamento.items():
    arquivo = next((a for a in candidatos if a in arquivos_disponiveis), None)
    if arquivo is None:
        raise FileNotFoundError(f"Não encontrei {candidatos} em {CAMINHO_INPUTS}.")
    bruto = (spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", False)
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .option("mode", "PERMISSIVE")
        .option("unescapedQuoteHandling", "STOP_AT_DELIMITER")
        .option("encoding", "UTF-8")
        .load(f"{CAMINHO_INPUTS}/{arquivo}"))
    # Nenhum trim, cast, filtro ou deduplicação ocorre nesta camada.
    (bruto.withColumn("ingestion_datetime", F.current_timestamp())
        .write.format("delta").mode("append")
        .saveAsTable(tabela("bronze", nome_tabela)))
    print(f"Arquivo {arquivo} gravado em {tabela('bronze', nome_tabela)}")

## Cotação do Banco Central

In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

sessao = requests.Session()
sessao.mount("https://", HTTPAdapter(max_retries=Retry(
    total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504]
)))
endpoint = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
)
parametros_api = {
    "@dataInicial": "'" + (DATA_INICIO-timedelta(days=15)).strftime("%m-%d-%Y") + "'",
    "@dataFinalCotacao": "'" + DATA_FIM.strftime("%m-%d-%Y") + "'",
    "$select": "dataHoraCotacao,cotacaoCompra",
    "$format": "json",
    "$orderby": "dataHoraCotacao asc"
}
registros = []
url = endpoint
while url:
    resposta = sessao.get(url, params=parametros_api, timeout=60)
    resposta.raise_for_status()
    conteudo = resposta.json()
    registros.extend(conteudo.get("value", []))
    url = conteudo.get("@odata.nextLink") or conteudo.get("odata.nextLink")
    parametros_api = None
if not registros:
    raise RuntimeError("A PTAX não retornou dados.")

esquema_api = T.StructType([
    T.StructField("dataHoraCotacao", T.StringType()),
    T.StructField("cotacaoCompra", T.StringType())
])
cotacao_bruta = spark.createDataFrame([
    (str(r["dataHoraCotacao"]), str(r["cotacaoCompra"])) for r in registros
], esquema_api)
(cotacao_bruta.withColumn("ingestion_datetime", F.current_timestamp())
    .write.format("delta").mode("append")
    .saveAsTable(tabela("bronze", "tb_cotacao_dolar")))
display(spark.table(tabela("bronze", "tb_cotacao_dolar")).orderBy(F.col("dataHoraCotacao").desc()).limit(10))